In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler

# 1. Data Loading and Cleaning
df = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
solar_data = df[['AT_solar_generation_actual']].dropna()

# 2. Normalization (MinMaxScaler)
# According to the guide, we bring the data into the [-1, 1] range
scaler = MinMaxScaler(feature_range=(-1, 1))
solar_data_scaled = scaler.fit_transform(solar_data.values)

# 3. Sliding Window Function
def create_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback)])
        y.append(data[i + lookback])
    return np.array(X), np.array(y)

# Since we use hourly data, we define a 24-hour (1 day) window
lookback = 24 
X, y = create_sequences(solar_data_scaled, lookback)

# 4. Splitting into Train and Test Sets
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# 5. Converting to Tensors
# PyTorch models require tensor inputs
X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_test_tensor = torch.from_numpy(y_test).float()

print("All Data Preparation Completed at Once!\n")
print(f"Training Data (X_train) Tensor Shape: {X_train_tensor.shape}")
print(f"Test Data (X_test) Tensor Shape: {X_test_tensor.shape}")